# Fine-tuning Stable Diffusion 2.1 con LoRA (alternativa leggera al fine-tuning completo)

Questo notebook affianca `03b_Finetuning_StableDiffusion2.1_filtered.ipynb` con una variante che addestra la U-Net di Stable Diffusion 2.1 usando **LoRA** (Low-Rank Adaptation, Hu et al. 2021) invece del fine-tuning completo: solo piccole matrici a basso rango vengono aggiornate (i pesi originali restano congelati), riducendo drasticamente i parametri allenabili, la VRAM richiesta e, potenzialmente, il costo energetico del training.

Domanda alla base del confronto: **a parità di dati, step di training e protocollo di selezione checkpoint, LoRA raggiunge una qualità generativa comparabile al fine-tuning completo di `03b`, con un costo computazionale nettamente inferiore?** È un'estensione della Domanda di ricerca D1 (qualità della generazione) con una lente di efficienza/sostenibilità (D3).

Per isolare l'effetto del **metodo** di fine-tuning (LoRA vs completo) dagli altri fattori, questo notebook riusa:

- **stessi dati** di `03b` (`data/processed/` + `data/real_augmented/`);
- **stesso modello base** SD2.1 originale (VAE e text encoder congelati, come in `03b`; nessun VAE adattato come in `03c`);
- **stessi step totali, stessa frequenza di checkpoint, stesso batch effettivo** di `03b` (8000 step, checkpoint ogni 500, batch 2 × grad-accum 4);
- **stessa pipeline di valutazione/generazione/filtro/test**, riutilizzata dinamicamente da `03b` (sezioni 5-10), con l'unica differenza tecnica che i checkpoint LoRA vengono caricati come adapter sopra la pipeline base invece che come U-Net completa (vedi sezione 5).

Cosa cambia rispetto a `03b`: script di training (`train_text_to_image_lora.py` invece di `train_text_to_image.py`), rango LoRA, learning rate più alto (prassi standard per LoRA) e assenza di 8-bit Adam (non supportato dallo script LoRA, e comunque meno necessario con così pochi parametri allenabili).

**Cartelle dedicate:**

- esperimento: `experiments/20260707_sd21_lora_finetuning/`
- risultati: `results/03d_finetuning_lora/`
- immagini sintetiche finali filtrate: `data/synthetic/fine_tuned_lora/{positive, negative}/`


## 1. Ambiente e configurazione

Le celle seguenti impostano l'ambiente di esecuzione, individuano la root del progetto, verificano PyTorch/GPU e definiscono una **singola configurazione condivisa** dall'intero notebook.

Rispetto a `03b`:

- `EXPERIMENT_NAME` e `RESULTS_03D_DIR` sono nuovi e isolano gli artefatti da quelli precedenti;
- viene aggiunta la dipendenza `peft` (richiesta da `train_text_to_image_lora.py` per gli adapter LoRA);
- vengono aggiunti `LORA_RANK` e un `LEARNING_RATE` più alto, dedicati al training LoRA;
- `MAX_TRAIN_STEPS`, `CHECKPOINTING_STEPS`, `CHECKPOINTS_TOTAL_LIMIT`, batch e gradient accumulation restano **identici a `03b`** per un confronto controllato.


In [ ]:
# Bootstrap dipendenze e import (equivalente 03b, EXPERIMENT_NAME aggiornato per 03d + peft per LoRA)
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime
from contextlib import contextmanager
import gc
import hashlib
import importlib
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import warnings
import zipfile

PROJECT_NAME = "MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None  # Esempio Colab: "/content/drive/MyDrive/MammoDiffusion"
EXPERIMENT_NAME = "20260707_sd21_lora_finetuning"
DIFFUSERS_REVISION = "3759fab56d3170a04d747e918a13e55fda6681e2"


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name or (
            (candidate / "data").is_dir() and (candidate / "notebooks").is_dir()
        ):
            return candidate

    fallback_candidates = [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]
    for candidate in fallback_candidates:
        if candidate.is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "Root di MammoDiffusion non trovata. Esegui il notebook dalla repository "
        "oppure imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR = EXPERIMENTS_DIR / EXPERIMENT_NAME
DIFFUSERS_REPO_DIR = EXPERIMENT_DIR / "diffusers_repo"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_INSTALL_PACKAGES = {
    "gdown": "gdown",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "IPython": "ipython",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "safetensors": "safetensors",
    "huggingface_hub": "huggingface_hub",
    "bitsandbytes": "bitsandbytes",
    "prdc": "prdc",
    "tensorboard": "tensorboard",
    "peft": "peft",  # richiesto da train_text_to_image_lora.py per gli adapter LoRA
}
missing_packages = [
    package_name
    for module_name, package_name in AUTO_INSTALL_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

missing_required = [
    module_name for module_name in ["torch"] if importlib.util.find_spec(module_name) is None
]
if missing_required:
    raise ImportError(
        "Dipendenze richieste non trovate: "
        + ", ".join(missing_required)
        + ". Installa PyTorch nell'ambiente prima di eseguire il notebook."
    )

if not (DIFFUSERS_REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/huggingface/diffusers", str(DIFFUSERS_REPO_DIR)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(DIFFUSERS_REPO_DIR), "checkout", DIFFUSERS_REVISION],
        check=True,
    )

TRAIN_SCRIPT = DIFFUSERS_REPO_DIR / "examples" / "text_to_image" / "train_text_to_image_lora.py"
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Script di training LoRA non trovato: {TRAIN_SCRIPT}")

current_revision = subprocess.check_output(
    [
        "git",
        "-c", f"safe.directory={DIFFUSERS_REPO_DIR}",
        "-C", str(DIFFUSERS_REPO_DIR),
        "rev-parse", "HEAD",
    ],
    text=True,
).strip()
if current_revision != DIFFUSERS_REVISION:
    print(f"ATTENZIONE: Diffusers è a {current_revision}, atteso {DIFFUSERS_REVISION}.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(DIFFUSERS_REPO_DIR)])
DIFFUSERS_SRC_DIR = DIFFUSERS_REPO_DIR / "src"
if str(DIFFUSERS_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(DIFFUSERS_SRC_DIR))
importlib.invalidate_caches()

for name in [
    "HF_HUB_VERBOSITY",
    "TRANSFORMERS_VERBOSITY",
    "DIFFUSERS_VERBOSITY",
    "ACCELERATE_LOG_LEVEL",
]:
    os.environ[name] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from generative_evaluator import GenerativeEvaluator
from eco_tracker import measure_sustainability
from IPython.display import display
from matplotlib.patches import Patch
from PIL import Image
from prdc import compute_prdc
from tensorboard.backend.event_processing import event_accumulator
from tqdm.auto import tqdm

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna")
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# Dataset condivisi dal progetto e configurazione dedicata a 03d
DATA_DIR = PROJECT_ROOT / "data"
ARCHIVES_DIR = DATA_DIR / "archives"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"

PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
AUGMENTED_DRIVE_ID = "1XRc0SxLEPP-zbMJApn4ruaiH8u_rDc-0"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
AUGMENTED_ZIP_PATH = ARCHIVES_DIR / "real_augmented.zip"

# Esperimento e repository Diffusers definiti nel bootstrap iniziale
SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
PRETRAINED_MODEL_DIR = EXPERIMENT_DIR / "pretrained_model" / "stable-diffusion-2-1-base"
PRETRAINED_MODEL_ZIP_PATH = EXPERIMENT_DIR / "archives" / "stable-diffusion-2-1-base.zip"
FORCE_MODEL_REDOWNLOAD = False

HF_CACHE_DIR = EXPERIMENT_DIR / "hf_cache"
SD_OUTPUT_DIR = EXPERIMENT_DIR / "model"

# Prompt condizionati dalle label (identici a 03a/03b/03c per confrontabilità)
POSITIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer positive, malignant finding, "
    "suspicious lesion, medical imaging"
)
NEGATIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer negative, no malignant finding, "
    "normal screening mammogram, medical imaging"
)

# Fine-tuning LoRA: step totali/checkpoint/batch identici a 03b, per confronto controllato.
# LEARNING_RATE e LORA_RANK sono invece specifici di LoRA (vedi sezione 4).
RESOLUTION = 512
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4  # 03b/03c (full fine-tune): 1e-5. LoRA ha molti meno parametri
                       # allenabili e converge meglio con un LR di un ordine superiore
                       # (prassi standard, vedi esempio ufficiale Diffusers LoRA).
LORA_RANK = 16         # dimensione delle matrici di aggiornamento a basso rango (default Diffusers: 4)
MAX_TRAIN_STEPS = 8000
CHECKPOINTING_STEPS = 500
CHECKPOINTS_TOTAL_LIMIT = 32
RESUME_FROM_CHECKPOINT = "latest"
TRAIN_SEED = 42

# Valutazione e generazione (identiche a 03b, cartelle dedicate)
N_EVAL_IMAGES_PER_CLASS = 100
N_VALIDATION_IMAGES_PER_CLASS = 73
N_TEST_IMAGES_PER_CLASS = 73
INFERENCE_STEPS = 100
EVAL_GUIDANCE_SCALE = 7.5
EVAL_SEED = 42
PRDC_NEAREST_K = 5
N_FINAL_IMAGES_PER_CLASS = 2722
FINAL_GENERATE_CLASSES = ["positive", "negative"]
N_SELECTED_PER_CLASS = 1361
RAW_MATCHED_SEED = 42
RAW_MATCHED_COUNT = N_SELECTED_PER_CLASS
RAW_MATCHED_ROOT = EXPERIMENT_DIR / "generated_images" / "raw_matched_1361"
NONBLACK_THRESHOLD = 10
FORCE_RECOMPUTE_VALIDATION_COMPARISON = False
FORCE_RECOMPUTE_FINAL_TEST = False

VALIDATION_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "val.csv"
TEST_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "test.csv"
TRAIN_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "train.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_03D_DIR = RESULTS_DIR / "03d_finetuning_lora"
METRICS_DIR = RESULTS_03D_DIR / "metrics"
PLOTS_DIR = RESULTS_03D_DIR / "plots"
ECOTRACKER_DIR = RESULTS_03D_DIR / "ecotracker"

EVAL_DIR = EXPERIMENT_DIR / "eval_checkpoints"
EVAL_METRICS_PATH = METRICS_DIR / "checkpoint_validation_metrics.json"
EVAL_SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_validation.jsonl"

FINAL_GEN_DIR = EXPERIMENT_DIR / "generated_images" / "final"
FINAL_NEG_DIR = FINAL_GEN_DIR / "negative"
FINAL_POS_DIR = FINAL_GEN_DIR / "positive"
FINAL_DIRS = {"negative": FINAL_NEG_DIR, "positive": FINAL_POS_DIR}
FINAL_CLASS_LABELS = {"negative": 0, "positive": 1}
RAW_MATCHED_DIRS = {
    class_name: RAW_MATCHED_ROOT / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
RAW_MATCHED_MANIFEST_PATHS = {
    class_name: RAW_MATCHED_ROOT / f"{class_name}_manifest.json"
    for class_name in FINAL_GENERATE_CLASSES
}

SYNTHETIC_DIR = DATA_DIR / "synthetic"
SYNTHETIC_LORA_DIR = SYNTHETIC_DIR / "fine_tuned_lora"
FILTERED_DIRS = {
    class_name: SYNTHETIC_LORA_DIR / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_REPORT_PATHS = {
    class_name: METRICS_DIR / f"filter_report_{class_name}_adaptive_mask.csv"
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_SUMMARY_PATHS = {
    class_name: METRICS_DIR / f"filter_summary_{class_name}_adaptive_mask.json"
    for class_name in FINAL_GENERATE_CLASSES
}

VALIDATION_COMPARISON_CSV = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.csv"
VALIDATION_COMPARISON_JSON = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.json"
FINAL_TEST_METRICS_PATH = METRICS_DIR / "final_test_metrics.json"
FINAL_TEST_METRICS_CSV = METRICS_DIR / "final_test_metrics.csv"
FINAL_TEST_RAW_MATCHED_METRICS_PATH = METRICS_DIR / "final_test_metrics_raw_matched_100_steps.json"
FINAL_TEST_RAW_MATCHED_METRICS_CSV = METRICS_DIR / "final_test_metrics_raw_matched_100_steps.csv"

SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_finetuning.jsonl"
FINAL_GENERATION_LOG = ECOTRACKER_DIR / "sustainability_generation.jsonl"
GENERATION_INFO_PATH = METRICS_DIR / "generation_info.json"

for directory in [
    ARCHIVES_DIR,
    DATA_AUG,
    EXPERIMENT_DIR,
    PRETRAINED_MODEL_ZIP_PATH.parent,
    HF_CACHE_DIR,
    SD_OUTPUT_DIR,
    EVAL_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    ECOTRACKER_DIR,
    FINAL_NEG_DIR,
    FINAL_POS_DIR,
    RAW_MATCHED_ROOT,
    *RAW_MATCHED_DIRS.values(),
    SYNTHETIC_LORA_DIR,
    *FILTERED_DIRS.values(),
]:
    directory.mkdir(parents=True, exist_ok=True)


def label_to_prompt(label):
    prompts = {0: NEGATIVE_PROMPT, 1: POSITIVE_PROMPT}
    try:
        return prompts[int(label)]
    except KeyError as exc:
        raise ValueError(f"Label non valida: {label}") from exc


print("Esperimento          :", EXPERIMENT_NAME)
print("Cartella esperimento :", EXPERIMENT_DIR)
print("Cartella risultati 3d:", RESULTS_03D_DIR)
print("Cartella sintetiche  :", SYNTHETIC_LORA_DIR)
print("LoRA rank            :", LORA_RANK)
print("Learning rate        :", LEARNING_RATE)
print("Inference step       :", INFERENCE_STEPS)


## 2. Preparazione e verifica dei dati

Identica a `03b`: verifica/download di `data/processed/` e `data/real_augmented/`, normalizzazione del metadata e `stage_training_dataset` per uno staging temporaneo compatibile con Diffusers. Copiata così com'è da `03b` per restare autosufficiente (nessun import di notebook durante il bootstrap), stessa convenzione già usata da `03c`.


In [ ]:
# Utility dati (copia autosufficiente da 03b, evita import ipynb durante il bootstrap)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ("train", "val", "test")
EXPECTED_LABELS = ("0", "1")


def count_images(directory):
    directory = Path(directory)
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    ) if directory.is_dir() else 0


def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archivio già presente:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Download non valido: {destination}")


def processed_dataset_ready(directory):
    directory = Path(directory)
    return all(
        count_images(directory / split / label) > 0
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )


def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} non trovata dentro {root}")


def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Dataset processed già pronto.")
        return

    download_zip(PROCESSED_DRIVE_ID, PROCESSED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_processed_extract_") as tmp:
        with zipfile.ZipFile(PROCESSED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, processed_dataset_ready, "processed")
        shutil.copytree(source_dir, DATA_PROCESSED_DIR, dirs_exist_ok=True)

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        raise FileNotFoundError("Dataset processed incompleto dopo l'estrazione.")


def augmented_dataset_ready():
    return (DATA_AUG / "metadata.csv").is_file() and count_images(DATA_AUG) > 0


def prepare_augmented_dataset():
    if augmented_dataset_ready():
        print("Dataset augmented già pronto.")
        return

    download_zip(AUGMENTED_DRIVE_ID, AUGMENTED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_augmented_extract_") as tmp:
        with zipfile.ZipFile(AUGMENTED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(
            tmp,
            lambda path: (path / "metadata.csv").is_file() and count_images(path) > 0,
            "augmented con metadata.csv",
        )
        DATA_AUG.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "metadata.csv", DATA_AUG / "metadata.csv")
        for image_path in source_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                destination = DATA_AUG / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)

    if not augmented_dataset_ready():
        raise FileNotFoundError("Dataset augmented incompleto dopo l'estrazione.")


def load_training_metadata(data_aug):
    metadata_path = Path(data_aug) / "metadata.csv"
    metadata = pd.read_csv(metadata_path).copy()
    required = {"file_name", "label"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Colonne mancanti in {metadata_path}: {sorted(missing)}")
    metadata["file_name"] = metadata["file_name"].astype(str).str.replace("\\", "/", regex=False)
    metadata["label"] = metadata["label"].astype(int)
    metadata["text"] = metadata["label"].map(label_to_prompt)
    return metadata


def is_valid_training_image(path):
    path = Path(path)
    return (
        path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def resolve_training_image_path(row):
    file_name = Path(str(row["file_name"]))
    label = str(int(row["label"]))
    source = str(row.get("source", "")).strip().lower()
    real_candidate = DATA_PROCESSED_DIR / "train" / label / file_name.name
    augmented_candidate = DATA_AUG / file_name.name

    if source == "real":
        candidates = [real_candidate]
    elif source in {"positive_augmentation", "augmentation", "augmented"}:
        candidates = [augmented_candidate]
    else:
        candidates = [augmented_candidate, real_candidate, PROJECT_ROOT / file_name]

    original_value = row.get("original_processed_path")
    if source == "real" and pd.notna(original_value) and str(original_value).strip():
        original = Path(str(original_value))
        if "data" in original.parts:
            candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
        candidates.append(original)

    for candidate in candidates:
        if is_valid_training_image(candidate):
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Immagine valida non trovata per file_name={row['file_name']}, source={source}. "
        f"Percorsi controllati: {checked}"
    )


def stage_training_dataset(metadata_df, staging_dir):
    """Copia ogni campione in uno staging temporaneo compatibile con HF imagefolder."""
    staging_dir = Path(staging_dir)
    staged = metadata_df.copy()
    staged_names = []

    for index, (_, row) in enumerate(staged.iterrows()):
        source_path = resolve_training_image_path(row)
        destination_name = f"image_{index:06d}{source_path.suffix.lower()}"
        shutil.copy2(source_path, staging_dir / destination_name)
        staged_names.append(destination_name)

    staged["file_name"] = staged_names
    staged.to_csv(staging_dir / "metadata.csv", index=False)

    if count_images(staging_dir) != len(staged):
        raise RuntimeError("Lo staging temporaneo non contiene tutti i campioni attesi.")
    return staged


In [ ]:
prepare_processed_dataset()
prepare_augmented_dataset()
metadata_df = load_training_metadata(DATA_AUG)

print("Campioni training:", len(metadata_df))
print("\nDistribuzione label:")
print(metadata_df["label"].value_counts().sort_index())
if "source" in metadata_df.columns:
    print("\nDistribuzione source:")
    print(metadata_df["source"].value_counts())

sample_df = metadata_df.sample(min(6, len(metadata_df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = resolve_training_image_path(row)
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f'label={row["label"]} | source={row.get("source", "n/a")}')
    axis.axis("off")
for axis in axes.flat[len(sample_df):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 3. Preparazione del modello base

Identica a `03b`: scarica/verifica il modello Stable Diffusion 2.1 originale (VAE e text encoder non toccati). A differenza di `03c`, qui **non** viene costruita alcuna copia con VAE adattato: LoRA viene confrontato con `03b` a parità di modello base, isolando l'effetto del solo metodo di fine-tuning della U-Net.


In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}


def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)


def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)


def create_model_weight_copies(model_dir):
    """Crea sempre copie fisiche con i nomi standard attesi da Diffusers."""
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Peso sorgente non trovato: {source_path}")
        shutil.copy2(source_path, target_path)


def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Modello Stable Diffusion 2.1 già pronto.")
        return PRETRAINED_MODEL_DIR.resolve()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "modello Diffusers")
        if PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Modello Stable Diffusion 2.1 incompleto dopo l'estrazione.")
    return PRETRAINED_MODEL_DIR.resolve()


LOCAL_MODEL_DIR = prepare_pretrained_model()
print("Modello locale:", LOCAL_MODEL_DIR)


In [ ]:
# Verifica del repository Diffusers locale e dello script di training LoRA.
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Script di training non trovato: {TRAIN_SCRIPT}")

print("Diffusers revision:", current_revision)
print("Script training LoRA:", TRAIN_SCRIPT)


## 4. Fine-tuning LoRA della U-Net

Usa `train_text_to_image_lora.py` (stesso repository/revisione Diffusers di `03b`/`03c`) invece di `train_text_to_image.py`. Differenze rispetto al comando di `03b`:

- solo le matrici LoRA (rango `LORA_RANK`) sono allenabili, iniettate nei layer di attenzione della U-Net; i pesi originali restano congelati;
- `--learning_rate` più alto (`LEARNING_RATE`) rispetto a `03b`/`03c`: con così pochi parametri allenabili, LoRA richiede tipicamente un learning rate di un ordine di grandezza superiore per convergere in un numero di step comparabile (prassi standard, vedi esempio ufficiale Diffusers);
- nessun `--use_8bit_adam`: lo script LoRA non lo supporta, ed è comunque meno necessario dato il numero ridotto di stati dell'ottimizzatore;
- `--validation_prompt` (singolare) invece di `--validation_prompts`: limite dello script LoRA di Diffusers, che accetta un solo prompt di validazione invece di uno per classe.

Step totali, frequenza di checkpoint, batch size e gradient accumulation restano identici a `03b` per un confronto controllato.


In [ ]:
def build_training_command(train_data_dir):
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--mixed_precision=fp16",
        "--num_processes=1",
        str(TRAIN_SCRIPT),
        "--pretrained_model_name_or_path", str(LOCAL_MODEL_DIR),
        "--train_data_dir", str(train_data_dir),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(RESOLUTION),
        "--center_crop",
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--gradient_checkpointing",
        "--max_train_steps", str(MAX_TRAIN_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--max_grad_norm", "1",
        "--rank", str(LORA_RANK),
        "--checkpointing_steps", str(CHECKPOINTING_STEPS),
        "--checkpoints_total_limit", str(CHECKPOINTS_TOTAL_LIMIT),
        "--validation_prompt", POSITIVE_PROMPT,
        "--num_validation_images", "2",
        "--validation_epochs", "2",
        "--seed", str(TRAIN_SEED),
        "--cache_dir", str(HF_CACHE_DIR),
        "--output_dir", str(SD_OUTPUT_DIR),
        "--report_to", "tensorboard",
        "--logging_dir", str(SD_OUTPUT_DIR / "logs"),
        "--dataloader_num_workers", "0",
    ]
    if RESUME_FROM_CHECKPOINT is not None:
        command.extend(["--resume_from_checkpoint", RESUME_FROM_CHECKPOINT])
    return command


def build_training_environment():
    environment = os.environ.copy()
    environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    conda_prefix = Path(sys.prefix)
    cuda_lib_paths = [
        conda_prefix / "lib",
        conda_prefix / "targets" / "x86_64-linux" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "cu13" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "nvjitlink" / "lib",
    ]
    existing = environment.get("LD_LIBRARY_PATH", "")
    environment["LD_LIBRARY_PATH"] = ":".join(
        [str(path) for path in cuda_lib_paths if path.exists()] + [existing]
    )
    return environment


run_label = f"finetune_sd21_lora_rank{LORA_RANK}_to_{MAX_TRAIN_STEPS}_steps"
if RESUME_FROM_CHECKPOINT is not None:
    run_label += f"_resume_{RESUME_FROM_CHECKPOINT}"
temporary_training = TemporaryDirectory(prefix="mammo_sd21_lora_train_copy_")
try:
    temporary_train_dir = Path(temporary_training.name)
    staged_metadata = stage_training_dataset(metadata_df, temporary_train_dir)
    command = build_training_command(temporary_train_dir)

    print("Staging temporaneo:", temporary_train_dir)
    print("Campioni copiati:", len(staged_metadata))
    print("Comando fine-tuning LoRA:\n", " ".join(map(str, command)))

    with measure_sustainability(label=run_label, sample_interval=0.5) as eco:
        process = subprocess.Popen(
            command,
            env=build_training_environment(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.wait()
finally:
    temporary_training.cleanup()
    print("Staging temporaneo eliminato.")

record = eco.metrics.to_dict()
record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_train_steps": MAX_TRAIN_STEPS,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resolution": RESOLUTION,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "source_metadata": str(DATA_AUG / "metadata.csv"),
    "staging_strategy": "temporary_copy",
    "output_dir": str(SD_OUTPUT_DIR),
    "returncode": process.returncode,
    "status": "completed" if process.returncode == 0 else "failed",
})
with SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
    handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Metriche sostenibilità:", eco.metrics)
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, command)


### Curva della loss di training


In [ ]:
event_files = sorted((SD_OUTPUT_DIR / "logs").rglob("events.out.tfevents.*"))
loss_rows = []

for event_file in event_files:
    try:
        accumulator = event_accumulator.EventAccumulator(
            str(event_file),
            size_guidance={"scalars": 0},
        )
        accumulator.Reload()
        scalar_tags = accumulator.Tags().get("scalars", [])
        preferred_tags = ["train_loss", "loss"]
        loss_tag = next((tag for tag in preferred_tags if tag in scalar_tags), None)
        if loss_tag is None:
            loss_tag = next((tag for tag in scalar_tags if tag.endswith("/loss")), None)
        if loss_tag is None:
            continue
        loss_rows.extend({
            "step": event.step,
            "loss": event.value,
            "wall_time": event.wall_time,
            "source": event_file.name,
            "tag": loss_tag,
        } for event in accumulator.Scalars(loss_tag))
    except Exception as exc:
        print(f"Evento TensorBoard non leggibile ({event_file.name}): {exc}")

if not loss_rows:
    print("Nessuna serie di loss TensorBoard disponibile: grafico non generato.")
else:
    df_loss = (
        pd.DataFrame(loss_rows)
        .sort_values(["step", "wall_time"])
        .drop_duplicates(subset="step", keep="last")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_loss["step"], df_loss["loss"], linewidth=1.2)
    ax.set(
        title="Loss di training LoRA registrata da TensorBoard",
        xlabel="Training step",
        ylabel="Loss",
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "train_loss.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Curva della loss salvata in:", PLOTS_DIR / "train_loss.png")


## 5. Valutazione dei checkpoint, generazione, filtro e test (riuso da 03b)

Come `03c`, questa sezione **importa dinamicamente le celle di codice del notebook `03b`** dalla sezione "## 5." in poi (selezione checkpoint, generazione RAW, dataset RAW matched, filtro adattivo, confronto validation, test finale, confronto 50 vs 100 step), per non duplicare centinaia di righe già testate.

**Unica differenza:** un checkpoint LoRA non è una pipeline completa come nel fine-tuning full — contiene solo l'adapter (`pytorch_lora_weights.safetensors`) da applicare sopra la pipeline base. Le funzioni `discover_checkpoints` e `load_pipeline_from_checkpoint` importate da `03b` vengono perciò **sovrascritte subito dopo l'import** con una versione compatibile con LoRA; tutte le altre funzioni riutilizzate (generazione, metriche FID/IS/PRDC, filtro, ecc.) restano invariate perché non dipendono dal formato del checkpoint.


In [ ]:
# Import dinamico delle utility valutazione/generazione/filtro/test da 03b (sezioni 5-10).
# Le variabili di configurazione 03d (SD_OUTPUT_DIR, FINAL_DIRS, ecc.) restano quelle definite
# sopra: le funzioni le leggono dal namespace globale, quindi punteranno automaticamente ai path 03d.

NB03B_PATH = NOTEBOOKS_DIR / "03b_Finetuning_StableDiffusion2.1_filtered.ipynb"
if not NB03B_PATH.is_file():
    raise FileNotFoundError(
        f"Notebook 03b non trovato in {NB03B_PATH}. "
        "Per eseguire 03d è necessaria la presenza delle utility di 03b nella stessa cartella."
    )

with NB03B_PATH.open(encoding="utf-8") as handle:
    nb03b_payload = json.load(handle)

# Estrae solo le celle di codice a partire dalla sezione 5 di 03b (valutazione checkpoint),
# che introduce generazione, filtro e test. Le celle 1-4 di 03b (bootstrap, dati, modello
# base, training) sono già coperte da 03d con la variante LoRA.
cells_to_reuse = []
in_reuse_zone = False
for cell in nb03b_payload["cells"]:
    source = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    if cell["cell_type"] == "markdown" and source.strip().startswith("## 5."):
        in_reuse_zone = True
        continue
    if not in_reuse_zone:
        continue
    if cell["cell_type"] != "code":
        continue
    cells_to_reuse.append(source)

print(f"Celle di codice riutilizzate da 03b: {len(cells_to_reuse)}")

# Le prime 3 celle riutilizzate sono, in ordine: configurazione selezione checkpoint,
# riferimenti reali temporanei, utility checkpoint/generazione/metriche. Quest'ultima
# definisce anche discover_checkpoints/load_pipeline_from_checkpoint, sovrascritte
# subito dopo con una versione compatibile con gli adapter LoRA.
for index, source in enumerate(cells_to_reuse[:3], start=1):
    print(f"\n===== 03b cell #{index} =====")
    exec(compile(source, f"<03b_cell_{index}>", "exec"), globals())


def discover_checkpoints(output_dir):
    """Come la versione di 03b, ma un checkpoint LoRA è valido se contiene
    pytorch_lora_weights.safetensors nella root, non una sottocartella unet/ completa."""
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "pytorch_lora_weights.safetensors").is_file():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)


def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    """Carica la pipeline base (pesi originali SD2.1) e applica sopra gli adapter LoRA
    del checkpoint, invece di sostituire l'intera U-Net come fa la versione full fine-tune."""
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.load_lora_weights(str(checkpoint_path))
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


for index, source in enumerate(cells_to_reuse[3:], start=4):
    print(f"\n===== 03b cell #{index} =====")
    exec(compile(source, f"<03b_cell_{index}>", "exec"), globals())


## 6. Confronto 03b (fine-tuning completo) vs 03d (LoRA)

Confronta le metriche finali sul test set reale (`03b_finetuning_filtered` vs `03d_finetuning_lora`, entrambe filtrate, 100 inference step, checkpoint selezionato con lo stesso protocollo di validation) e il costo del training (tempo, energia, CO₂ tracciati con `eco_tracker`, dimensione del checkpoint salvato). Nessuna metrica viene ricalcolata: si leggono solo gli artefatti già salvati da entrambe le pipeline.


In [ ]:
RESULTS_03B_DIR = RESULTS_DIR / "03b_finetuning_filtered"
final_test_03b_path = RESULTS_03B_DIR / "metrics" / "final_test_metrics.json"
final_test_03d_path = FINAL_TEST_METRICS_PATH


def load_final_test_metrics(path, tag):
    if not Path(path).is_file():
        print(f"[{tag}] metriche finali test non disponibili: {path}")
        return None
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


metrics_03b = load_final_test_metrics(final_test_03b_path, "03b")
metrics_03d = load_final_test_metrics(final_test_03d_path, "03d")

if metrics_03b is None or metrics_03d is None:
    print("Confronto qualità 03b vs 03d non disponibile (metriche mancanti su almeno una pipeline).")
else:
    METRIC_ORDER = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
    LOWER_IS_BETTER = {"FID"}

    def rows_from(payload, tag):
        rows = []
        for class_name, metrics in payload.get("per_class", {}).items():
            if class_name == "average":
                continue
            row = {"pipeline": tag, "class": class_name}
            for metric in METRIC_ORDER:
                row[metric] = metrics.get(metric)
            rows.append(row)
        return rows

    df_compare = pd.DataFrame(rows_from(metrics_03b, "03b_full_finetune") + rows_from(metrics_03d, "03d_lora"))
    compare_csv = METRICS_DIR / "comparison_03b_vs_03d.csv"
    df_compare.to_csv(compare_csv, index=False)
    print("Riepilogo qualità generativa salvato in:", compare_csv)
    print(df_compare.to_string(index=False))

    delta_rows = []
    for class_name in df_compare["class"].unique():
        row_full = df_compare[(df_compare["pipeline"] == "03b_full_finetune") & (df_compare["class"] == class_name)]
        row_lora = df_compare[(df_compare["pipeline"] == "03d_lora") & (df_compare["class"] == class_name)]
        if row_full.empty or row_lora.empty:
            continue
        entry = {"class": class_name}
        for metric in METRIC_ORDER:
            baseline = row_full.iloc[0][metric]
            candidate = row_lora.iloc[0][metric]
            if pd.isna(baseline) or pd.isna(candidate):
                entry[metric] = None
                continue
            raw_delta_pct = (candidate - baseline) / baseline * 100 if baseline != 0 else float("nan")
            entry[metric] = round(-raw_delta_pct if metric in LOWER_IS_BETTER else raw_delta_pct, 2)
        delta_rows.append(entry)

    df_delta = pd.DataFrame(delta_rows)
    print("\nDelta % (positivo = LoRA migliore del fine-tuning completo):")
    print(df_delta.to_string(index=False))


def count_safetensors_params(path):
    """Conta i parametri di un file .safetensors leggendo solo gli shape, senza caricare i tensori."""
    import math
    from safetensors import safe_open
    total = 0
    with safe_open(str(path), framework="pt") as handle:
        for key in handle.keys():
            shape = handle.get_slice(key).get_shape()
            total += math.prod(shape) if shape else 1
    return total


def sum_completed_sustainability(log_path, label_prefix):
    """Somma le run completate di un log eco_tracker il cui label inizia con label_prefix
    (un training ripreso più volte produce più righe nello stesso file)."""
    log_path = Path(log_path)
    if not log_path.is_file():
        return None
    totals = {"elapsed_seconds": 0.0, "energy_kwh": 0.0, "co2_kg": 0.0, "n_runs": 0}
    with log_path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if record.get("status") == "completed" and str(record.get("label", "")).startswith(label_prefix):
                totals["elapsed_seconds"] += float(record.get("elapsed_seconds", 0.0))
                totals["energy_kwh"] += float(record.get("energy_kwh", 0.0))
                totals["co2_kg"] += float(record.get("co2_kg", 0.0))
                totals["n_runs"] += 1
    return totals if totals["n_runs"] else None


EXPERIMENT_NAME_03B = "20260611_sd21_rsna_mlo_512_inference_100_steps"  # deve combaciare con EXPERIMENT_NAME in 03b

lora_cost = sum_completed_sustainability(SUSTAINABILITY_LOG, "finetune_sd21_lora_")
full_ft_cost = sum_completed_sustainability(
    RESULTS_03B_DIR / "ecotracker" / "sustainability_finetuning.jsonl", "finetune_sd21_to_"
)

lora_weights_path = (
    Path(BEST_CHECKPOINT) / "pytorch_lora_weights.safetensors"
    if metrics_03d is not None and "BEST_CHECKPOINT" in globals()
    else None
)
lora_size_mb = lora_weights_path.stat().st_size / (1024 ** 2) if lora_weights_path and lora_weights_path.is_file() else None
lora_params = count_safetensors_params(lora_weights_path) if lora_weights_path and lora_weights_path.is_file() else None

full_ft_generation_info = RESULTS_03B_DIR / "metrics" / "generation_info.json"
full_unet_size_mb = None
if full_ft_generation_info.is_file():
    with full_ft_generation_info.open(encoding="utf-8") as handle:
        best_checkpoint_03b = json.load(handle).get("best_checkpoint")
    if best_checkpoint_03b:
        full_unet_path = (
            PROJECT_ROOT / "experiments" / EXPERIMENT_NAME_03B / "model"
            / best_checkpoint_03b / "unet" / "diffusion_pytorch_model.safetensors"
        )
        if full_unet_path.is_file():
            full_unet_size_mb = full_unet_path.stat().st_size / (1024 ** 2)

print("\n=== Costo ed efficienza del training (LoRA vs fine-tuning completo) ===")
if lora_size_mb is not None:
    extra = f" | {lora_params / 1e6:.2f}M parametri allenati" if lora_params else ""
    print(f"Checkpoint LoRA (best)       : {lora_size_mb:.1f} MB{extra}")
if full_unet_size_mb is not None:
    print(f"Checkpoint U-Net completo    : {full_unet_size_mb:.1f} MB (03b, stesso formato safetensors)")
if lora_size_mb is not None and full_unet_size_mb is not None:
    print(f"Riduzione dimensione checkpoint: {(1 - lora_size_mb / full_unet_size_mb) * 100:.1f}%")
if lora_cost is not None:
    print(
        f"Training LoRA            : {lora_cost['elapsed_seconds'] / 3600:.2f} h | "
        f"{lora_cost['energy_kwh']:.4f} kWh | {lora_cost['co2_kg']:.4f} kg CO2 "
        f"({lora_cost['n_runs']} run completate)"
    )
if full_ft_cost is not None:
    print(
        f"Training completo (03b)  : {full_ft_cost['elapsed_seconds'] / 3600:.2f} h | "
        f"{full_ft_cost['energy_kwh']:.4f} kWh | {full_ft_cost['co2_kg']:.4f} kg CO2 "
        f"({full_ft_cost['n_runs']} run completate)"
    )
if lora_cost is not None and full_ft_cost is not None and full_ft_cost["elapsed_seconds"] > 0:
    saving_pct = (1 - lora_cost["elapsed_seconds"] / full_ft_cost["elapsed_seconds"]) * 100
    print(f"Risparmio tempo di training: {saving_pct:.1f}%")

efficiency_summary = {
    "lora_checkpoint_size_mb": lora_size_mb,
    "lora_trainable_params": lora_params,
    "full_finetune_checkpoint_size_mb": full_unet_size_mb,
    "lora_training_cost": lora_cost,
    "full_finetune_training_cost": full_ft_cost,
}
efficiency_path = METRICS_DIR / "efficiency_comparison_03b_vs_03d.json"
with efficiency_path.open("w", encoding="utf-8") as handle:
    json.dump(efficiency_summary, handle, indent=2, ensure_ascii=False)
print("\nRiepilogo efficienza salvato in:", efficiency_path)


## Note

Notebook non ancora eseguito: verrà lanciato dopo il completamento di `03c` (fine-tuning U-Net su VAE adattato) e dei notebook `04b3`/`04d`, secondo la pianificazione corrente del progetto. Se il fine-tuning LoRA si rivelasse competitivo con `03b` a un costo nettamente inferiore, è un candidato naturale per addestrare ulteriori varianti senza ripetere l'intero costo di un fine-tuning completo.
